In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print('length of dataset in characters:', len(text))

length of dataset in characters: 1115394


In [3]:
#lets look at first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
chars = (sorted(list(set(text))))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
#create a mapping from chars to int
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] ##encoder: take a string, output a list of intergers
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: take a list of intergers, output a string

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [21]:
# lets now encode the entire dataet and store it into a torch.tensor
# !pip install torch
import torch
max_iters = 3000
learning_rate = 1e-2
eval_interval = 300
eval_iters = 200

In [22]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [23]:
# now lets split the data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [24]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [25]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[: t + 1]
    target = y[t]
    print (f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [26]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel
block_size = 8 #what is the max contect len for predictions
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

def get_batch(split):
    #generate a small batch of data of inputs x an dtargets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x,y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----------------------')

for b in range(batch_size): #batch dim
    for t in range(block_size): #time dim
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----------------------
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
whe

In [27]:
print(xb) #input for transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [28]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)


class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        #each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):

        #idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) #(B,T,C)
        B,T,C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        
        return logits, loss

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


In [29]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out




class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        #each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        #idx and targets are noth (B,T) tensor of integers
        logits = self.token_embedding_table(idx) #(B,T,C)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context

        for _ in range(max_new_tokens):
            #get the predictions
            logits, loss = self(idx)
            # focus on the last time step
            logits = logits[:,-1, :] # becomes (B,C)
            #apply softmax to get probability
            probs = F.softmax(logits, dim = 1) #(b,c)
            #sample from the disctribution
            idx_next = torch.multinomial(probs, num_samples = 1) #(B,1)
            #append sampledd index to the rinning sequence
            idx = torch.cat((idx, idx_next), dim = 1) #(B, T+1)
        return idx

model = BigramLanguageModel(vocab_size)
m = model.to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.6630, grad_fn=<NllLossBackward0>)

v?vqKGh
txYXcDRSW&jxn;P Kuvbv
Bl:KuMWi;JYlFahlK3u.M,QV;oky3wcxoYPQpbpb
r3Nc 'MQHQ uOfxoxwXIvSE-$&a$K


In [32]:
# create a PyTorch optimizer

optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)

In [34]:
# batch_size = 32

# for steps in range(10000):

#     # sample a batch of data
#     xb, yb = get_batch('train')

#     #evaluate the loss
#     logits, loss = m(xb, yb)
#     optimizer.zero_grad(set_to_none = True)
#     loss.backward()
#     optimizer.step()

#     print(loss.item())
# max_iters = 10000
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']: 4f}, val loss {losses['val']: .4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    #evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

step 0: train loss  4.592515, val loss  4.6303
step 300: train loss  4.408099, val loss  4.4054
step 600: train loss  4.187620, val loss  4.1826
step 900: train loss  3.998093, val loss  4.0093
step 1200: train loss  3.843267, val loss  3.8525
step 1500: train loss  3.703279, val loss  3.7056
step 1800: train loss  3.552284, val loss  3.5917
step 2100: train loss  3.417233, val loss  3.4612
step 2400: train loss  3.329500, val loss  3.3500
step 2700: train loss  3.232409, val loss  3.2504
step 3000: train loss  3.139850, val loss  3.1655
step 3300: train loss  3.104742, val loss  3.1007
step 3600: train loss  3.001844, val loss  3.0425
step 3900: train loss  2.956247, val loss  2.9629
step 4200: train loss  2.911809, val loss  2.9234
step 4500: train loss  2.845034, val loss  2.8647
step 4800: train loss  2.830227, val loss  2.8486
step 5100: train loss  2.791535, val loss  2.7936
step 5400: train loss  2.757043, val loss  2.7557
step 5700: train loss  2.722238, val loss  2.7310
step 6

In [35]:
context = torch.zeros((1,1), dtype = torch.long, device= device)
print(decode(m.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens =500)[0].tolist()))


Ad be tas g thatrth, ysu; daven, swh as
PE:ZUnd, tir pssisaur I:
ivorde ar n ath t! woroGLI berek-is t out ads a RINonofiWQHod T:
By ave oryund y.
l ie bss.
TI h, g hiz3pe seBHOTo tsie my ar:

nd we KIUCran, l thesthitshe d hndw:
Y mye as st?PaizqJvis; sw thoomy
Y m wsal haig as3. y, patobst w hanuolo avVIOKMe he lugarth nchofino;OKa he.
Whicorat al menoud. Y:
NNs weose.
Eve the st ppKof meerenger:
GLmakl f b' lere st

A wintit. ow tche fusobulWIUCADqFo pigllh s,
THNRRI erd sloory tr, s ledes,
d


A mathematical trick in self attention

In [36]:
# consider the following toy example

torch.manual_seed(1337)
B,T,C = 4,8,2 #batch, time, chsnnels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [38]:
# We want x[b,t] = mean_{i<=t} x [b,i]

xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] #t,C
        xbow[b,t] = torch.mean(xprev, 0)
        

In [39]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [41]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [44]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
b = torch.randint(0, 10, (3,2)).float()
c = a @ b
print('a = ')
print(a)
print('b = ')
print(b)
print('c =')
print(c)

a = 
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
b = 
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c =
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


In [48]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim= True)
xbow2 = wei @ x # (T,T) @ (B,T,C)
torch.allclose(xbow, xbow2)

False